## Creating Embeddings

In [0]:
df = spark.read.csv("dbfs:/product_details/products_AT_de_AT.csv", sep=";", header=True, inferSchema=True)
df.createOrReplaceTempView('product_details_at')

df = df.filter(df.Langbeschreibung.isNotNull())
display(df)

In [0]:
df = spark.sql("""
        select p.*, i.item_id, i.category_caption_level_2, i.family 
        from product_details_at p
        inner join curated.dim_item_detail_pim i 
        on p.identifier = i.identifier
        """)

In [0]:
from pyspark.sql.functions import regexp_replace, concat_ws, initcap, expr

df = df.withColumn("Name AX", initcap(df["Name AX"]))
df = df.withColumnRenamed("Name AX", "Name_ax")

df = df.withColumn("Inhaltsstoffe_new",expr("array_join(regexp_extract_all(Inhaltsstoffe, '<Description>(.*?)</Description>'), ', ')"))

df = df.withColumn("Warnungen_new", regexp_replace(df.Warnungen, r'<CautionId>.*?</CautionId>', ''))
df = df.withColumn("Warnungen_new", regexp_replace(df.Warnungen_new, r'<[^>]*>', ''))

df = df.withColumnRenamed("URL-Prefix", "url_prefix")
df = df.withColumn("category", concat_ws(" ", df["url_prefix"], df['category_caption_level_2'], df['family']))


df = df.withColumn("Langbeschreibung_new", regexp_replace(df.Langbeschreibung, r'<[^>]*>', ''))
df = df.withColumn("descriptions", concat_ws(" ", df["Langbeschreibung_new"], df['Keywords'], df['ABDA Warengruppe Code'], df['seo_slug']))
df = df.withColumn("descriptions", regexp_replace(df.descriptions, r'\s{3,}', '  '))
display(df.limit(10))

# Format 2

In [0]:
import threading
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import (
    StructType, StructField, ArrayType, FloatType
)
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

Getting Embeddings

In [0]:
def get_model():
    # Cache the model per executor to avoid reloading
    if not hasattr(get_model, "model"):
        from sentence_transformers import SentenceTransformer
        get_model.model = SentenceTransformer('distiluse-base-multilingual-cased-v2')
    return get_model.model

In [0]:

# Define output schema: one field per input column with an "_embedding" suffix
embedding_schema = StructType([
    StructField("Name_AX_embedding", ArrayType(FloatType())),
    StructField("Inhaltsstoffe_new_embedding", ArrayType(FloatType())),
    StructField("Warnungen_new_embedding", ArrayType(FloatType())),
    StructField("category_embedding", ArrayType(FloatType())),
    StructField("descriptions_embedding", ArrayType(FloatType()))
])


In [0]:
@pandas_udf(embedding_schema)
def compute_all_embeddings_udf(name_ax: pd.Series,
                               inhaltsstoffe: pd.Series,
                               warnungen: pd.Series,
                               category: pd.Series,
                               descriptions: pd.Series) -> pd.DataFrame:
    # Prepare each column: fill missing values and convert to string list
    name_ax_list = name_ax.fillna("").astype(str).tolist()
    inhaltsstoffe_list = inhaltsstoffe.fillna("").astype(str).tolist()
    warnungen_list = warnungen.fillna("").astype(str).tolist()
    category_list = category.fillna("").astype(str).tolist()
    descriptions_list = descriptions.fillna("").astype(str).tolist()

    # Get the cached model
    model = get_model()
    
    # Function to compute embeddings for a list of texts using parallel processing
    def encode_texts(text_list):
        num_threads = 4  # adjust based on available cores per executor
        # Split texts into chunks for each thread
        chunk_size = len(text_list) // num_threads + 1
        chunks = [text_list[i:i + chunk_size] for i in range(0, len(text_list), chunk_size)]
        with ThreadPoolExecutor(max_workers=num_threads) as executor:
            # Each thread encodes its chunk concurrently
            results = list(executor.map(
                lambda chunk: model.encode(chunk, batch_size=128, show_progress_bar=False),
                chunks
            ))
        # Flatten the list while preserving order
        embeddings = [emb for sublist in results for emb in sublist]
        # Convert each embedding (a numpy array) to a list
        return [emb.tolist() for emb in embeddings]

    # Compute embeddings for each column
    name_ax_emb = encode_texts(name_ax_list)
    inhaltsstoffe_emb = encode_texts(inhaltsstoffe_list)
    warnungen_emb = encode_texts(warnungen_list)
    category_emb = encode_texts(category_list)
    descriptions_emb = encode_texts(descriptions_list)
    
    # Build a DataFrame with a column for each embedding output
    return pd.DataFrame({
        "Name_AX_embedding": name_ax_emb,
        "Inhaltsstoffe_new_embedding": inhaltsstoffe_emb,
        "Warnungen_new_embedding": warnungen_emb,
        "category_embedding": category_emb,
        "descriptions_embedding": descriptions_emb
    })


In [0]:
df = df.repartition(48)


In [0]:
df_with_embeddings = df.withColumn(
    "embeddings",
    compute_all_embeddings_udf(
        col("Name_AX"),
        col("Inhaltsstoffe_new"),
        col("Warnungen_new"),
        col("category"),
        col("descriptions")
    )
)

# Expand the struct into separate columns and drop the temporary "embeddings" column
for field in embedding_schema.fields:
    df_with_embeddings = df_with_embeddings.withColumn(field.name, col("embeddings." + field.name))
df_with_embeddings = df_with_embeddings.drop("embeddings").cache()

In [0]:
display(df_with_embeddings.limit(10))

In [0]:
import re

def clean_column_name(name):
    return re.sub(r'[ ,;{}()\n\t=]', ' ', name)

for col_name in df_with_embeddings.columns:
    df_with_embeddings = df_with_embeddings.withColumnRenamed(col_name, clean_column_name(col_name))

display(df_with_embeddings.limit(10))

In [0]:
df_with_embeddings.write.format("delta").mode("overwrite").saveAsTable("datascience.at_product_details_individual_embeddings")